# AI Image Detection — Training Notebook (Local)

Trains **EfficientNet-B0** (CNN) and **DeiT-Tiny** (Transformer) on the **ArtiFact** dataset for cross-generator AI-image detection. Outputs best-validation checkpoints + ONNX exports for both models.

**Local setup (one-time):**
1. Install deps:
   ```
   pip install torch torchvision timm==1.0.11 kagglehub pandas pillow scikit-learn onnxruntime
   ```
   For CUDA support, install torch from pytorch.org with the build matching your CUDA version.
2. Place your Kaggle API token at `~/.kaggle/kaggle.json` (Linux/Mac) or `%USERPROFILE%\.kaggle\kaggle.json` (Windows). `chmod 600` it on Unix.
3. Make sure you have **70GB+ free disk** on an SSD for the full ArtiFact download.

**Subset vs full run:** Set `MAX_IMAGES_PER_GENERATOR = None` for a full run. Start with a subset (e.g. 5000) to verify the pipeline before committing a long run.

Checkpoints land in `./checkpoints/` next to this notebook.

## 1. Verify environment

In [ ]:
import torch, sys
print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no CUDA detected. Training will be very slow on CPU.')

## 2. Set up checkpoint directory

In [ ]:
import os

CKPT_DIR = './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {os.path.abspath(CKPT_DIR)}')

## 3. Verify Kaggle credentials & download dataset

Make sure `~/.kaggle/kaggle.json` exists locally (or `%USERPROFILE%\.kaggle\kaggle.json` on Windows). `kagglehub` reads it automatically and caches the dataset to `~/.cache/kagglehub/`.

In [ ]:
# kagglehub auto-reads ~/.kaggle/kaggle.json — no upload needed locally.
# Just verify it's there before attempting download.
from pathlib import Path

kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
if not kaggle_json.exists():
    raise FileNotFoundError(
        f'No Kaggle token found at {kaggle_json}. '
        'Download one from https://www.kaggle.com/settings → Create New API Token.'
    )
print(f'Kaggle token found at {kaggle_json}')

In [ ]:
# Download ArtiFact via kagglehub. Resumes if partially downloaded.
# NOTE: full dataset is ~70GB+. First download is slow.
# Cached to ~/.cache/kagglehub/ — subsequent runs are instant.
import kagglehub

DATASET_SLUG = 'awsaf49/artifact-dataset'   # adjust if you're using a different mirror
DATA_ROOT = kagglehub.dataset_download(DATASET_SLUG)
print(f'Dataset at: {DATA_ROOT}')

# If you've already downloaded ArtiFact manually, comment out the lines above
# and uncomment the line below, pointing to your local copy:
# DATA_ROOT = '/path/to/your/artifact'

# List what's actually there so we can map generator folders correctly
for entry in sorted(os.listdir(DATA_ROOT))[:50]:
    full = os.path.join(DATA_ROOT, entry)
    kind = 'DIR ' if os.path.isdir(full) else 'FILE'
    print(f'  {kind}  {entry}')

In [ ]:
# Download ArtiFact via kagglehub. Resumes if partially downloaded.
# NOTE: full dataset is ~70GB+. First download is slow (30–90 min on Colab).
import kagglehub

DATASET_SLUG = 'awsaf49/artifact-dataset'   # adjust if you're using a different mirror
DATA_ROOT = kagglehub.dataset_download(DATASET_SLUG)
print(f'Dataset at: {DATA_ROOT}')

# List what's actually there so we can map generator folders correctly
for entry in sorted(os.listdir(DATA_ROOT))[:50]:
    full = os.path.join(DATA_ROOT, entry)
    kind = 'DIR ' if os.path.isdir(full) else 'FILE'
    print(f'  {kind}  {entry}')

In [ ]:
import torch

# ── Data layout ─────────────────────────────────────────
TRAIN_GENERATORS = [
    'stable_diffusion',
    'progan',
    'stylegan',
    'dalle',
]
HOLDOUT_GENERATORS = [
    'midjourney',
    'glide',
]

# Metadata column names — adjust if the CSVs use different headers
IMG_COL   = 'image_path'
LABEL_COL = 'target'   # ArtiFact uses 'target'; some forks use 'label'

# ── Training hyperparameters ────────────────────────────
MAX_IMAGES_PER_GENERATOR = 5000   # set to None for full dataset
IMG_SIZE    = 224
BATCH_SIZE  = 32                  # drop to 16 if you OOM on 6GB VRAM
NUM_EPOCHS  = 20
LR          = 1e-4
PATIENCE    = 4                   # early stopping
VAL_SPLIT   = 0.1
NUM_WORKERS = 4                   # bump higher if you have many CPU cores
SEED        = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import torch

# ── Data layout ─────────────────────────────────────────
TRAIN_GENERATORS = [
    'stable_diffusion',
    'progan',
    'stylegan',
    'dalle',
]
HOLDOUT_GENERATORS = [
    'midjourney',
    'glide',
]

# Metadata column names — adjust if the CSVs use different headers
IMG_COL   = 'image_path'
LABEL_COL = 'target'   # ArtiFact uses 'target'; some forks use 'label'

# ── Training hyperparameters ────────────────────────────
MAX_IMAGES_PER_GENERATOR = 5000   # set to None for full dataset
IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_EPOCHS  = 20
LR          = 1e-4
PATIENCE    = 4                   # early stopping
VAL_SPLIT   = 0.1
NUM_WORKERS = 2
SEED        = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 5. Dataset class & transforms

In [ ]:
import random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

random.seed(SEED)
torch.manual_seed(SEED)

class ArtiFactDataset(Dataset):
    """Loads images from a list of generator folders. Each folder has a metadata.csv."""
    def __init__(self, data_root, generators, transform=None, max_per_gen=None):
        self.transform = transform
        self.samples = []   # list of (image_path, label, generator)

        for gen in generators:
            gen_dir   = os.path.join(data_root, gen)
            meta_path = os.path.join(gen_dir, 'metadata.csv')
            if not os.path.isdir(gen_dir):
                print(f'[WARN] missing folder: {gen_dir}')
                continue
            if not os.path.exists(meta_path):
                print(f'[WARN] no metadata.csv in {gen}, skipping.')
                continue

            df = pd.read_csv(meta_path)
            if max_per_gen is not None and len(df) > max_per_gen:
                df = df.sample(n=max_per_gen, random_state=SEED).reset_index(drop=True)

            for _, row in df.iterrows():
                img_path = os.path.join(gen_dir, str(row[IMG_COL]))
                self.samples.append((img_path, int(row[LABEL_COL]), gen))

        print(f'Loaded {len(self.samples):,} samples from {len(generators)} generator(s).')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, gen = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            # fallback for corrupted images — return a zero tensor with the same label
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, label, gen


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## 6. Build train / val / holdout loaders

In [ ]:
# Training pool — generators seen during training
full_train = ArtiFactDataset(
    DATA_ROOT, TRAIN_GENERATORS,
    transform=train_tf, max_per_gen=MAX_IMAGES_PER_GENERATOR,
)

val_size   = int(len(full_train) * VAL_SPLIT)
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
# Validation should use eval transforms — swap on the underlying dataset copy
val_ds.dataset = ArtiFactDataset(
    DATA_ROOT, TRAIN_GENERATORS,
    transform=eval_tf, max_per_gen=MAX_IMAGES_PER_GENERATOR,
)

# Holdout — generators NEVER seen during training
holdout_ds = ArtiFactDataset(
    DATA_ROOT, HOLDOUT_GENERATORS,
    transform=eval_tf, max_per_gen=MAX_IMAGES_PER_GENERATOR,
)

def collate(batch):
    imgs, labels, gens = zip(*batch)
    return torch.stack(imgs), torch.tensor(labels), list(gens)

train_loader   = DataLoader(train_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
val_loader     = DataLoader(val_ds,     batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

print(f'Train:   {len(train_ds):,}')
print(f'Val:     {len(val_ds):,}')
print(f'Holdout: {len(holdout_ds):,}')

## 7. Model factory (timm)

In [ ]:
import timm
import torch.nn as nn

MODELS = {
    'efficientnet_b0': 'efficientnet_b0',
    'deit_tiny':       'deit_tiny_patch16_224',
}

def build_model(name: str) -> nn.Module:
    timm_name = MODELS[name]
    model = timm.create_model(timm_name, pretrained=True, num_classes=2)
    return model.to(DEVICE)

## 8. Train / eval loops

In [ ]:
import time, copy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(imgs)
            loss = criterion(logits, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / n, accuracy_score(all_labels, all_preds)


def train_model(name: str):
    print(f'\n{"="*60}\nTRAINING: {name}\n{"="*60}')
    model     = build_model(name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_loss = float('inf')
    best_state    = None
    best_epoch    = -1
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None)
        scheduler.step()

        print(f'[{name}] epoch {epoch:02d}/{NUM_EPOCHS}  '
              f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  '
              f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}  '
              f'({time.time()-t0:.0f}s)')

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state    = copy.deepcopy(model.state_dict())
            best_epoch    = epoch
            epochs_no_improve = 0
            ckpt_path = os.path.join(CKPT_DIR, f'{name}_best.pt')
            torch.save({
                'model_state_dict': best_state,
                'epoch': epoch,
                'val_loss': vl_loss,
                'val_acc':  vl_acc,
                'model_name': MODELS[name],
                'img_size': IMG_SIZE,
                'num_classes': 2,
            }, ckpt_path)
            print(f'    ↳ new best, saved → {ckpt_path}')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'    ↳ early stopping at epoch {epoch} (best was {best_epoch}).')
                break

    model.load_state_dict(best_state)
    return model

## 9. Evaluation (overall + per-generator on holdout)

In [ ]:
def evaluate(model, loader, label):
    model.eval()
    all_preds, all_labels, all_gens = [], [], []
    with torch.no_grad():
        for imgs, labels, gens in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            preds = model(imgs).argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
            all_gens.extend(gens)

    print(f'\n── {label} ──')
    acc = accuracy_score(all_labels, all_preds)
    pre = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    print(f'overall  acc={acc:.4f}  prec={pre:.4f}  rec={rec:.4f}  f1={f1:.4f}')

    # per-generator breakdown
    df = pd.DataFrame({'pred': all_preds, 'label': all_labels, 'gen': all_gens})
    print('\nper-generator accuracy:')
    for g, sub in df.groupby('gen'):
        print(f'  {g:24s}  acc={accuracy_score(sub.label, sub.pred):.4f}  (n={len(sub)})')
    return df

## 10. Run training for both models

In [ ]:
effnet_model = train_model('efficientnet_b0')
_ = evaluate(effnet_model, val_loader,     'EfficientNet-B0 — validation (seen generators)')
_ = evaluate(effnet_model, holdout_loader, 'EfficientNet-B0 — holdout (unseen generators)')

In [ ]:
# Free VRAM before training the second model
import gc
del effnet_model
gc.collect()
torch.cuda.empty_cache()

deit_model = train_model('deit_tiny')
_ = evaluate(deit_model, val_loader,     'DeiT-Tiny — validation (seen generators)')
_ = evaluate(deit_model, holdout_loader, 'DeiT-Tiny — holdout (unseen generators)')

## 11. Verify checkpoints

Both checkpoints are now in your Drive at `MyDrive/ai_detection_ckpts/`. They survive runtime disconnects and can be loaded for ONNX export or browser-extension packaging.

In [ ]:
for f in sorted(os.listdir(CKPT_DIR)):
    full = os.path.join(CKPT_DIR, f)
    print(f'  {f:40s}  {os.path.getsize(full)/1e6:.1f} MB')

# Quick load sanity check
for name in MODELS:
    ckpt = torch.load(os.path.join(CKPT_DIR, f'{name}_best.pt'), map_location='cpu', weights_only=False)
    print(f'\n{name}:  epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}  val_acc={ckpt["val_acc"]:.4f}')

## 12. Export checkpoints to ONNX (for JavaScript browser extension)

Exports both best checkpoints to ONNX format compatible with ONNX Runtime Web. Uses opset 17 for browser compatibility. Dynamic batch axis included.

In [ ]:
import torch

ONNX_OPSET = 17   # safe target for ONNX Runtime Web

def export_to_onnx(name: str):
    ckpt_path = os.path.join(CKPT_DIR, f'{name}_best.pt')
    onnx_path = os.path.join(CKPT_DIR, f'{name}_best.onnx')

    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = build_model(name).cpu()
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

    torch.onnx.export(
        model, dummy, onnx_path,
        export_params=True,
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={
            'input':  {0: 'batch'},
            'logits': {0: 'batch'},
        },
    )
    print(f'  exported → {onnx_path}  ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

for name in MODELS:
    export_to_onnx(name)

## 13. Verify ONNX exports with onnxruntime

If these run cleanly and produce shape `(1, 2)`, the files are valid for ONNX Runtime Web.

In [ ]:
!pip install -q onnxruntime

import onnxruntime as ort
import numpy as np

for name in MODELS:
    onnx_path = os.path.join(CKPT_DIR, f'{name}_best.onnx')
    sess = ort.InferenceSession(onnx_path)
    dummy = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    out = sess.run(None, {'input': dummy})
    print(f'{name:20s}  output shape: {out[0].shape}   (expect (1, 2))')